# 🧠 NeuroDSL – Stirling Numbers & Bell Numbers

This notebook demonstrates **NeuroDSL**, a Julia DSL for declarative,
recursive computation graphs with **automatic memoisation**.

We compute **Stirling numbers of the second kind** and the **Bell number** $ B_6 $,
then visualise the entire computation graph interactively.

---

## 📐 Recurrence

Stirling numbers of the second kind \( S(n,k) \) satisfy:

$$
S(n,k) =
\begin{cases}
1 & \text{if } n = 0,\; k = 0,\\[4pt]
0 & \text{if } k = 0 \text{ or } k > n,\\[4pt]
k \cdot S(n-1,k) + S(n-1,k-1) & \text{otherwise}.
\end{cases}
$$

The **Bell number**  $\displaystyle B_n = \sum_{k=1}^{n} S(n,k) $ counts all partitions of an $ n$-element set.

For $ n = 6 $, $ B_6 = 203 $.

---


# Installation (run once)

In [1]:
using NeuroDSL, Printf

┌ Warning: Circular dependency detected. Precompilation will be skipped for:
│   cuSPARSE [b26da814-b3bc-49ef-b0ee-c816305aa060]
│   ChainRulesCoreExt [8ccf546c-b5ad-5491-b9b6-b62e2a4db961]
│   GPUArraysSparseArraysExt [555d850b-e4ac-5d9d-bfa0-17bb432b4efb]
│   ChainRules [082447d4-558c-5d27-93f4-14fc19e9eca2]
│   NVML [611af6d1-644e-4c5d-bd58-854d7d1254b9]
│   GPUArrays [0c68f7d7-f131-5f86-a1c3-88cf8149b2d7]
│   JLD2Ext [3dbd0623-ce14-52d8-8601-bc177a3b211d]
│   KernelAbstractions [63c18a36-062a-441e-b654-da1e3ab1ce7c]
│   ZygoteExt [3f48baa7-5843-56cc-8004-b8b725de387b]
│   FluxCUDAExt [dd41ee52-2073-581e-92e8-26baf003f19a]
│   cuSOLVER [887afef0-6a32-4de5-add4-7827692ba8fc]
│   CUDAExt [9e0aca61-9665-56cf-9642-d947ef6fc392]
│   StructArraysAdaptExt [f04e5bcb-ab32-5a64-8b64-c2cc4abec66e]
│   OneHotArrays [0b1bfda6-eb8a-41d2-88d8-f5af5cad476f]
│   CUDACore [bd0ed864-bdfe-4181-a5ed-ce625a5fdea2]
│   ZygoteDistancesExt [5865c103-18d1-586a-9b11-010bbc2260a8]
│   MLUtilsExt [447e3217-c189

# Define custom operators with @defop

In [9]:

@defop scale_add out = attrs[:factor] * inputs[1] + inputs[2]
@defop identity  out = inputs[1]
@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

✅ Op :scale_add registered
✅ Op :identity registered
✅ Op :nsum registered


# Build the recursive Stirling graph

In [10]:
function build_stirling_graph(g, n)
    builder = @neuro g ns=:stirling operators=[:scale_add] begin
        @node one  = 1.0
        @node zero = 0.0

        @rule stirling(n::Int, k::Int) = begin
            if n == 0 && k == 0
                :one
            elseif k == 0 || k > n
                :zero
            else
                scale_add(factor=k, stirling(n-1, k), stirling(n-1, k-1))
            end
        end

        # Compute all S(n,k) for k=1..n
        all_terms = [stirling(n, i) for i in 1:n]
        @node bell = nsum(all_terms...)
    end
    return builder
end

n = 6
g = NeuroGraph(namespace=:stirling, device=Backend.CPUDevice())
builder = build_stirling_graph(g, n)

GraphBuilder(NeuroGraph(Dict{Symbol, Dict{Symbol, GraphNode{Float32}}}(:stirling => Dict(:stirling_27 => GraphNode{Float32}(:stirling_27, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any}(:label => "stirling(5, 2)", :is_rule => true), Symbol[], nothing), :stirling_25 => GraphNode{Float32}(:stirling_25, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any}(:label => "stirling(4, 2)", :is_rule => true), Symbol[], nothing), :scale_add_14 => GraphNode{Float32}(:scale_add_14, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any}(:label => "scale_add_14 = 1·stirling(4, 1) + zero"), Symbol[], nothing), :scale_add_8 => GraphNode{Float32}(:scale_add_8, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any}(:label => "scale_add_8 = 1·stirling(2, 1) + zero"), Symbol[], nothing), :scale_add_40 => GraphNode{Float32}(:scale_add_40, nothing, nothing, false, false, Quantom, false, :stirling, Dict{Symbol, Any

# Execute forward pass

In [11]:
log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :bell; ctx_store=ctx, namespace=:stirling, log=log)

bell_val = Array(NeuroDSL.node(g, :bell; namespace=:stirling).value)[]
@printf "Bell number B_%d = %d\n" n Int(bell_val)

Bell number B_6 = 203


 # Generate interactive HTML trace

In [12]:
save_interactive_graph(g, log, "stirling_bell_6.html";
                       title = "Stirling & Bell numbers (n=$n)")

✅ Interactive Trace exported → stirling_bell_6.html


In [13]:
# Windows
run(`cmd /c start stirling_bell_6.html`)



Process(`cmd /c start stirling_bell_6.html`, ProcessExited(0))

In [2]:
using NeuroDSL, Printf

# ── Opérateur personnalisé : combinaison linéaire à deux poids ──
@defop linear2 (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    w1 = attrs[:w1]
    w2 = attrs[:w2]
    @. out = w1 * inputs[1] + w2 * inputs[2]
end

# Opérateurs de base
@defop nsum (dev, out, inputs, attrs, out_sym, nd, ctx) -> begin
    copyto!(out, inputs[1])
    for i in 2:length(inputs)
        out .+= inputs[i]
    end
end

@defop identity out = inputs[1]

# ── Construction du graphe ──────────────────────────────────
function build_eulerian_graph(g, n)
    builder = @neuro g ns=:euler operators=[:linear2] begin
        @node one  = 1.0
        @node zero = 0.0

        @rule eulerian(n::Int, k::Int) = begin
            if n == 0 && k == 0
                :one
            elseif k == 0 || k > n
                :zero
            else
                linear2(w1 = n - k + 1, w2 = k,
                        eulerian(n-1, k-1), eulerian(n-1, k))
            end
        end

        # Somme de tous les A(n,k) pour k=1..n
        all_terms = [eulerian(n, i) for i in 1:n]
        @node sum_euler = nsum(all_terms...)
    end
    return builder
end

# ── Exécution ───────────────────────────────────────────────
n = 5   # A(5,k) et somme = 5! = 120
g = NeuroGraph(namespace=:euler, device=Backend.CPUDevice())
builder = build_eulerian_graph(g, n)

log = ExecutionLog()
ctx = CtxStore()
NeuroDSL.demand!(g, :sum_euler; ctx_store=ctx, namespace=:euler, log=log)

sum_val = Array(NeuroDSL.node(g, :sum_euler; namespace=:euler).value)[]
println("Sum of Eulerian numbers A($n,k) = ", sum_val)
println("Expected n! = ", factorial(n))

# ── Visualisation interactive ───────────────────────────────
save_interactive_graph(g, log, "eulerian_n5.html";
                       title = "Eulerian numbers A($n,k) with NeuroDSL")

✅ Op :linear2 registered
✅ Op :nsum registered
✅ Op :identity registered
Sum of Eulerian numbers A(5,k) = 120.0
Expected n! = 120
✅ Interactive Trace exported → eulerian_n5.html
